# Chapter 10 - Part 2: Regression, classification, and honest evaluation

## Learning objectives

1. Audit experimental labels and structures while preserving their row alignment.
2. Explain why related molecules and learned preprocessing can cause information leakage.
3. Compare small classical models and baselines using training-only cross-validation.
4. Assess the selected models once on held-out chemical groups, using appropriate metrics.
5. Distinguish a demonstration score from evidence of practical applicability.

**Run from the repository root using [Readme.md](Readme.md).** This notebook is standalone and reads two tracked CSV files offline. It audits the full files, then limits feature calculation to **800 log D records and 600 BBBP records**, using fixed seeds chosen before model evaluation. Models use three training folds and small, fixed candidate sets; there is no exhaustive search. Outputs go to `outputs/chapter10_part2/`.

### Start here: a model is a rule learned from examples

This notebook studies two different questions: **how large is a measured log D?** (regression) and **which of two benchmark labels applies?** (classification). A descriptor or fingerprint is the input, not the answer.

| Term | Plain-language meaning |
|---|---|
| Fit or train | Adjust a model's numerical parameters using examples with known labels. |
| Loss/error | A stated measure of disagreement between a prediction and a label. |
| Hyperparameter | A setting chosen around training, such as a penalty strength or tree depth. |
| Validation fold | Examples temporarily hidden while selecting settings within the training pool. |
| Test set | A reserved final assessment after those choices are frozen. |
| Baseline | A simple reference rule, such as always predicting the training median. |
| Generalization | How well a fitted rule transfers to the particular new examples of interest. |

**First pass:** follow the target definitions, split diagram, baseline comparisons, and final diagnostic figures. The code that preserves row identity and fits preprocessing inside folds is part of the scientific method: it prevents a seemingly good score from answering the wrong question.

**Decision exercise declared before fitting:** after the final assessment, use the frozen BBBP ranking to examine an illustrative inspection budget of **20% of the held-out records**. This will be a retrospective explanation of the existing result, not another threshold-selection or model-selection step.

## 1. Define the target before the algorithm

| Local dataset | Target used here | Interpretation and provenance |
|---|---|---|
| `datasets/Lipophilicity.csv` (4200 rows) | `lipophilicity` | Experimental octanol/water **log D at pH 7.4**, a dimensionless base-10 logarithm. The local SMILES/target pairs match the MoleculeNet/DeepChem Lipophilicity source, whose target column is named `exp`. |
| `datasets/BBBP.csv` (2050 rows) | `p_np`, 0 or 1 | The distributed BBBP benchmark's negative/positive blood-brain-barrier penetration label. This is a curated literature-derived classification endpoint, not a measured continuous permeability coefficient or a universal clinical outcome. |

The Lipophilicity target is **not** the calculated `RDKit MolLogP` descriptor and is not neutral-species log P. Ionization, pH, assay context, and measurement uncertainty matter. The BBBP paper and distributed benchmark have different record counts: this lesson identifies the actual 2050-row distributed file, rather than claiming it is an exact reconstruction of the original publication's curated study.

Sources: [MoleculeNet paper](https://doi.org/10.1039/C7SC02664A), [DeepChem dataset documentation](https://deepchem.readthedocs.io/en/latest/api_reference/moleculenet.html), [Martins et al., original BBB study](https://doi.org/10.1021/ci300124c), and [local provenance notes](datasets/README.md). No downloads occur during execution. The educational BBBP model does not establish brain exposure, efficacy, safety, or suitability of a compound for treatment.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
import sklearn
from sklearn.base import clone
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, StratifiedGroupKFold
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve

OUT = Path('outputs/chapter10_part2')
OUT.mkdir(parents=True, exist_ok=True)
SEED = 2026
INSPECTION_FRACTION = 0.20  # Declared before fitting; used only for final interpretation.
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
print('RDKit:', rdBase.rdkitVersion, '| scikit-learn:', sklearn.__version__)

## 2. Audit first, with a declared curation policy

We keep the source row number, input SMILES, canonical **isomeric** SMILES, and target together in one table. The source CSV files are never modified.

- Reject missing/nonfinite labels and failed RDKit parses; BBBP labels must be exactly 0 or 1.
- Identify canonical groups with different labels. Exclude the entire conflicting group for this teaching analysis because these small files lack enough assay context to resolve it. This does **not** establish that any underlying measurement is wrong. For continuous measurements, differing replicate values may be entirely legitimate; a research workflow should preserve and model their experimental context.
- Exclude disconnected structures instead of silently deleting salts or choosing a component. This changes the population represented by the lesson.
- Keep the first source row in each remaining, consistently labeled canonical group. No tautomer or charge neutralization is performed; canonicalization does not make these chemical choices for us.
- Finally sample a bounded subset without looking at model scores. Audits describe the full local files; reported model scores describe only these curated subsets.

The individual audit flag counts can overlap. The final retained count follows the stated sequence. Parsing messages are suppressed only inside the audit; every rejected record and its reason are saved explicitly.

In [ ]:
def audit_dataset(path, target, task, limit):
    raw = pd.read_csv(path)
    if not {'smiles', target}.issubset(raw.columns):
        raise ValueError(f'Missing required columns in {path}')
    rows = []
    with rdBase.BlockLogs():
        for source_row, row in raw.iterrows():
            value = pd.to_numeric(row[target], errors='coerce')
            valid_label = bool(pd.notna(value) and np.isfinite(value))
            if task == 'classification':
                valid_label = valid_label and value in (0, 1)
            smiles = row['smiles']
            mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) and smiles.strip() else None
            valid_structure = mol is not None and mol.GetNumAtoms() > 0
            canonical = Chem.MolToSmiles(mol, isomericSmiles=True) if valid_structure else None
            rows.append({'source_row': source_row, 'smiles': smiles, 'y': value,
                         'canonical': canonical, 'mol': mol, 'valid_label': valid_label,
                         'valid_structure': valid_structure,
                         'disconnected': valid_structure and len(Chem.GetMolFrags(mol)) != 1})
    audit = pd.DataFrame(rows)
    parseable = audit[audit['valid_structure'] & audit['valid_label']]
    label_counts = parseable.groupby('canonical')['y'].nunique()
    conflicts = set(label_counts[label_counts > 1].index)
    audit['conflicting_group'] = audit['canonical'].isin(conflicts)
    eligible = audit['valid_structure'] & audit['valid_label'] & ~audit['disconnected'] & ~audit['conflicting_group']
    audit['retained_unique'] = False
    unique_rows = audit.loc[eligible].drop_duplicates('canonical', keep='first')
    audit.loc[unique_rows.index, 'retained_unique'] = True
    selected = unique_rows.sample(n=min(limit, len(unique_rows)), random_state=SEED).sort_values('source_row').copy()
    audit['selected'] = audit['source_row'].isin(selected['source_row'])
    summary = {
        'source_rows': len(raw), 'invalid_structures': int((~audit['valid_structure']).sum()),
        'invalid_labels': int((~audit['valid_label']).sum()),
        'canonical_duplicate_extra_rows': int(parseable.duplicated('canonical').sum()),
        'conflicting_canonical_groups': len(conflicts),
        'rows_in_conflicting_groups': int(audit['conflicting_group'].sum()),
        'disconnected_rows': int(audit['disconnected'].sum()),
        'retained_unique_rows': len(unique_rows), 'selected_rows': len(selected),
    }
    audit.drop(columns='mol').to_csv(OUT / f'{path.stem}_audit.csv', index=False)
    assert selected['canonical'].is_unique
    return selected.reset_index(drop=True), summary

def scaffold_group(mol):
    # Same exact ring/linker scaffold -> same group; stereochemistry is ignored for grouping.
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    return scaffold if scaffold else 'ACYCLIC'

In [ ]:
lipo_path = Path('datasets/Lipophilicity.csv')
reg_data, reg_audit = audit_dataset(lipo_path, 'lipophilicity', 'regression', limit=800)
reg_data['group'] = [scaffold_group(mol) for mol in reg_data['mol']]
display(pd.Series(reg_audit, name='Lipophilicity audit').to_frame())

In [ ]:
bbbp_path = Path('datasets/BBBP.csv')
cls_data, cls_audit = audit_dataset(bbbp_path, 'p_np', 'classification', limit=600)
cls_data['y'] = cls_data['y'].astype(int)
cls_data['group'] = [scaffold_group(mol) for mol in cls_data['mol']]
display(pd.Series(cls_audit, name='BBBP audit').to_frame())

## 3. Split chemistry, then learn transformations

Randomly splitting individual records often puts close analogs in both sets. Here each **exact Murcko scaffold** (ring systems plus connecting linkers under RDKit's definition) belongs entirely to one side. Stereoisomers of the same scaffold stay together. All acyclic structures share one `ACYCLIC` group; this coarse choice can produce an uneven split and a difficult domain shift. We report actual sizes, not a promised 75:25 ratio of molecules.

This is one explicit chemical grouping policy. Related structures with different exact scaffolds can still cross splits; salts and tautomers are not fully standardized. A scaffold holdout is not automatically a realistic time split, an external validation set, or a guarantee of novelty. Sources: [RDKit scaffold API](https://www.rdkit.org/docs/source/rdkit.Chem.Scaffolds.MurckoScaffold.html), [Bemis and Murcko](https://doi.org/10.1021/jm9602928), and [scikit-learn grouped splitting](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data).

**Evaluation protocol, fixed before fitting:** reserve 25% of scaffold groups as the test set. Use three group-disjoint folds *within the training portion* to select a model. Each fold fits its own imputer, scaler, or variance filter inside a `Pipeline`. Refit the selected pipeline on all training data; evaluate the reserved test set once. The initial data-quality audit uses labels to resolve duplicates, but no test score, fitted transformation, or test error is used for model selection.

In [ ]:
def outer_split(data):
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
    train, test = next(splitter.split(data, groups=data['group']))
    assert set(data.iloc[train]['group']).isdisjoint(data.iloc[test]['group'])
    assert set(data.iloc[train]['canonical']).isdisjoint(data.iloc[test]['canonical'])
    return train, test

reg_train, reg_test = outer_split(reg_data)
cls_train, cls_test = outer_split(cls_data)
split_rows = []
for name, data, train, test in [('log D', reg_data, reg_train, reg_test),
                                ('BBBP', cls_data, cls_train, cls_test)]:
    data['split'] = 'test'
    data.loc[train, 'split'] = 'train'
    for split, indices in [('train', train), ('test', test)]:
        split_rows.append({'dataset': name, 'split': split, 'molecules': len(indices),
                           'scaffold_groups': data.iloc[indices]['group'].nunique(),
                           'acyclic_molecules': int((data.iloc[indices]['group'] == 'ACYCLIC').sum())})
display(pd.DataFrame(split_rows))

## 4. Regression: predict measured log D at pH 7.4

We calculate nine named 2D descriptors. Donors and acceptors are labeled correctly. `MolLogP` is a calculated feature related to the endpoint, not a copy of the measured target: including an independently calculated physical estimate is legitimate. The label stays in its original log D scale; we never repeatedly inverse-transform it.

Fixed descriptor formulas can be evaluated for every selected structure independently. Learned preprocessing is different: fitting a scaler, imputer, feature selector, or PCA on the full dataset before splitting leaks information. Even unsupervised preprocessing belongs inside cross-validation. See [scikit-learn's worked leakage examples](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).

In [ ]:
descriptor_functions = {
    'MolWt': Descriptors.MolWt, 'heteroatoms': Descriptors.NumHeteroatoms,
    'rings': Descriptors.RingCount, 'HBA': Descriptors.NumHAcceptors,
    'HBD': Descriptors.NumHDonors, 'FractionCSP3': Descriptors.FractionCSP3,
    'TPSA': Descriptors.TPSA, 'MolLogP': Descriptors.MolLogP, 'MolMR': Descriptors.MolMR,
}
X_reg = np.array([[function(mol) for function in descriptor_functions.values()]
                  for mol in reg_data['mol']], dtype=float)
y_reg = reg_data['y'].to_numpy(dtype=float)
assert X_reg.shape == (len(reg_data), len(descriptor_functions))
assert np.isfinite(X_reg).all() and np.isfinite(y_reg).all()
Xr_train, yr_train = X_reg[reg_train], y_reg[reg_train]
gr_train = reg_data.iloc[reg_train]['group'].to_numpy()
reg_folds = list(GroupKFold(n_splits=3).split(Xr_train, yr_train, groups=gr_train))
for fit, validate in reg_folds:
    assert set(gr_train[fit]).isdisjoint(gr_train[validate])
print('Training feature matrix:', Xr_train.shape, '| named features:', list(descriptor_functions))

### See where each molecule is allowed to contribute

The diagram uses the **actual log D split and folds** defined above. Each column is one selected molecule, displayed with training molecules first and grouped by their assigned scaffold. Every fold learns its preprocessing and model from blue columns, measures validation error on gold columns, and leaves the gray test columns untouched. A molecule can validate one fold and train another; it cannot be both within the same fold.

After settings are selected, the final pipeline is refitted on the entire training pool. The reserved test labels then answer a separate assessment question. Group sizes differ, so validation blocks need not contain equal numbers of molecules.

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

fold_assignment = np.zeros((len(reg_folds), len(reg_data)), dtype=int)
for fold_number, (fit, validate) in enumerate(reg_folds):
    assert set(fit).isdisjoint(validate) and len(fit) + len(validate) == len(reg_train)
    fold_assignment[fold_number, reg_train[fit]] = 1
    fold_assignment[fold_number, reg_train[validate]] = 2
assert np.all(fold_assignment[:, reg_test] == 0)
assert np.all((fold_assignment[:, reg_train] == 2).sum(axis=0) == 1)
display_order = np.r_[sorted(reg_train, key=lambda i: (reg_data.loc[i, 'group'], int(i))),
                       sorted(reg_test, key=lambda i: (reg_data.loc[i, 'group'], int(i)))]
fig, ax = plt.subplots(figsize=(10, 3.2), layout='constrained')
colors = ['#dedede', '#28788e', '#dda64a']
ax.imshow(fold_assignment[:, display_order], aspect='auto', interpolation='nearest',
           cmap=ListedColormap(colors), norm=BoundaryNorm([-0.5, 0.5, 1.5, 2.5], 3))
ax.axvline(len(reg_train)-0.5, color='black', lw=1.5)
ax.set(yticks=range(len(reg_folds)), yticklabels=[f'Fold {i+1}' for i in range(len(reg_folds))],
       xticks=[(len(reg_train)-1)/2, len(reg_train)+(len(reg_test)-1)/2],
       xticklabels=[f'Training pool: {len(reg_train)} molecules', f'Reserved test: {len(reg_test)} molecules'],
       title='Model selection happens entirely inside the training pool')
ax.legend(handles=[Patch(color=color, label=label) for color, label in zip(colors,
           ['Reserved test', 'Fit this fold', 'Validate this fold'])],
           loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False)
fig.savefig(OUT / 'actual_grouped_validation_folds.png')
plt.show()

### Baseline and bounded model selection

The **median baseline** ignores structure and predicts the training median, a natural constant baseline for mean absolute error (MAE). Ridge regression adds an L2 penalty to a linear model; scaling makes its penalty comparable across differently sized features. We try only two fixed penalty strengths. A small random forest provides one nonlinear comparison, averaging 32 bounded-depth trees. Forests do not require feature scaling.

Cross-validation selects the lowest mean validation MAE from these three candidates. Negative MAE is scikit-learn's maximization convention; the table changes its sign back. The fold standard deviation describes variation across three particular folds, **not a confidence interval**. These folds can differ in size and chemical difficulty, and selection makes the best validation score optimistic as a final performance estimate.

Bias is systematic error across hypothetical training samples; variance describes sensitivity to the sampled training data. A training-versus-validation gap can warn of overfitting, but one such gap does not measure that formal decomposition. Regularization and a simpler model can help; neither guarantees generalization.

In [ ]:
reg_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', Ridge()),
])
reg_candidates = [
    {'model': [Ridge()], 'model__alpha': [1.0, 10.0]},
    {'scaler': ['passthrough'],
     'model': [RandomForestRegressor(n_estimators=32, max_depth=6, min_samples_leaf=3,
                                     random_state=SEED, n_jobs=1)]},
]
reg_search = GridSearchCV(reg_pipeline, reg_candidates, scoring='neg_mean_absolute_error',
                          cv=reg_folds, n_jobs=1, refit=True, return_train_score=True, error_score='raise')
reg_search.fit(Xr_train, yr_train)
reg_cv = pd.DataFrame(reg_search.cv_results_)
reg_cv_table = pd.DataFrame({
    'candidate': [f'Ridge alpha={p["model__alpha"]:g}' if 'model__alpha' in p else 'Random forest (32 trees)'
                  for p in reg_cv['params']],
    'training_MAE': -reg_cv['mean_train_score'],
    'validation_MAE': -reg_cv['mean_test_score'],
    'validation_fold_SD': reg_cv['std_test_score'],
})
reg_baseline = DummyRegressor(strategy='median')
baseline_reg_cv = cross_validate(reg_baseline, Xr_train, yr_train, cv=reg_folds,
                                 scoring='neg_mean_absolute_error', error_score='raise')
display(reg_cv_table.round(3))
print(f'Median baseline validation MAE: {-baseline_reg_cv["test_score"].mean():.3f}')
print('Selected:', reg_cv_table.iloc[reg_search.best_index_]['candidate'])

### Final regression evaluation: one held-out prediction per model

$$\mathrm{MAE}=\frac{1}{n}\sum_i |y_i-\hat y_i|,\qquad
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_i(y_i-\hat y_i)^2},\qquad
R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y_{\mathrm{test}})^2}.$$

MAE and RMSE are reported in **log D units**; RMSE weights large errors more strongly. An absolute error of one log unit corresponds to a tenfold ratio error in D. Predictive $R^2$ can be negative: it then performs worse in squared error than the constant test-set mean used only as a mathematical reference. Our deployable baseline uses the *training* median, so its test $R^2$ need not be zero. Metrics on a constant target require special care because the denominator of $R^2$ is zero.

The test plots below are final diagnostics, not instructions to choose another model after seeing the result. Further tuning would require another independent assessment. Source: [scikit-learn regression metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics).

In [ ]:
reg_model = reg_search.best_estimator_  # already refitted on all training rows
reg_baseline.fit(Xr_train, yr_train)
yr_test = y_reg[reg_test].copy()
reg_prediction = reg_model.predict(X_reg[reg_test])
reg_baseline_prediction = reg_baseline.predict(X_reg[reg_test])
reg_metrics = pd.DataFrame([
    {'model': name, 'MAE_logD': mean_absolute_error(yr_test, pred),
     'RMSE_logD': root_mean_squared_error(yr_test, pred), 'R2': r2_score(yr_test, pred)}
    for name, pred in [('selected model', reg_prediction), ('training median', reg_baseline_prediction)]
]).set_index('model')
assert np.isfinite(reg_prediction).all()
reg_results = reg_data.iloc[reg_test][['source_row', 'canonical', 'group', 'y']].copy()
reg_results['prediction'] = reg_prediction
reg_results['residual_observed_minus_predicted'] = yr_test - reg_prediction
reg_results.to_csv(OUT / 'logD_test_predictions.csv', index=False)
display(reg_metrics.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.3), layout='constrained')
limits = [min(yr_test.min(), reg_prediction.min()) - 0.2, max(yr_test.max(), reg_prediction.max()) + 0.2]
axes[0].scatter(yr_test, reg_prediction, s=22, alpha=0.65)
axes[0].plot(limits, limits, '--', color='black', lw=1, label='perfect prediction')
axes[0].set(xlabel='Observed log D (pH 7.4)', ylabel='Predicted log D (pH 7.4)',
            xlim=limits, ylim=limits, title='Held-out scaffold groups')
axes[0].legend(fontsize=9)
axes[1].scatter(reg_prediction, yr_test - reg_prediction, s=22, alpha=0.65)
axes[1].axhline(0, color='black', linestyle='--', lw=1)
axes[1].set(xlabel='Predicted log D (pH 7.4)', ylabel='Observed minus predicted (log D units)',
            title='Residuals: inspect systematic errors')
fig.savefig(OUT / 'regression_test_diagnostics.png')
plt.show()

### Chemical similarity is a diagnostic, not an error bar

After final scoring, compare each held-out compound with its nearest training fingerprint. High similarity can coexist with large error; low similarity may signal extrapolation but does not determine an error bound. This descriptor model was not trained on the fingerprints used for this diagnostic. Neither this plot nor a hand-picked Tanimoto cutoff establishes a calibrated applicability domain.

In [ ]:
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024,
                                                   includeChirality=True, countSimulation=False)
reg_fps = [morgan.GetFingerprint(mol) for mol in reg_data['mol']]
training_fps = [reg_fps[i] for i in reg_train]
nearest_similarity = np.array([max(DataStructs.BulkTanimotoSimilarity(reg_fps[i], training_fps))
                                for i in reg_test])
reg_results['nearest_training_Tanimoto'] = nearest_similarity
reg_results.to_csv(OUT / 'logD_test_predictions.csv', index=False)
fig, ax = plt.subplots(figsize=(6.8, 4.1), layout='constrained')
ax.scatter(nearest_similarity, np.abs(yr_test - reg_prediction), s=24, alpha=0.65)
ax.set(xlabel='Largest Morgan Tanimoto to a training molecule', ylabel='Absolute error (log D units)',
       xlim=(0, 1), title='Final diagnostic; no threshold was used to select results')
fig.savefig(OUT / 'similarity_and_error.png')
plt.show()

## 5. Classification: predict the benchmark's BBBP label

The representation is a 1024-bit Morgan fingerprint, radius 2, with chirality enabled. These features are binary indicators, not one-hot molecule identifiers or measured probabilities. A variance filter removes columns constant in each training fold; the whole operation stays in a `Pipeline`. We do not fit a full-dataset PCA or scaler before splitting. Binary features do not require scaling for this chosen implementation.

**Logistic regression** models a linear score followed by a logistic function to estimate class-1 probability. Despite its name, this is a classification model. We compare only `C=0.1` and `C=1.0`; smaller C means stronger regularization. The `prior` dummy classifier predicts the training majority class and returns the same training class probabilities for every molecule.

We choose C using mean validation ROC-AUC. Three-fold `StratifiedGroupKFold` attempts to balance classes while keeping entire chemical groups together; both aims cannot always be satisfied perfectly. We check the actual folds and do not retry seeds to obtain a favorable score. The final classification threshold is fixed at **0.5 before evaluation**, with class 1 as the positive class. Sources: [logistic regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression), [StratifiedGroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html), and [dummy estimators](https://scikit-learn.org/stable/modules/model_evaluation.html#dummy-estimators).

In [ ]:
X_cls = np.stack([morgan.GetFingerprintAsNumPy(mol) for mol in cls_data['mol']])
y_cls = cls_data['y'].to_numpy(dtype=int)
Xc_train, yc_train = X_cls[cls_train], y_cls[cls_train]
gc_train = cls_data.iloc[cls_train]['group'].to_numpy()
assert set(np.unique(X_cls)) <= {0, 1}
cls_folds = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED).split(
    Xc_train, yc_train, groups=gc_train))
fold_rows = []
for fold_number, (fit, validate) in enumerate(cls_folds, start=1):
    assert set(gc_train[fit]).isdisjoint(gc_train[validate])
    assert len(np.unique(yc_train[fit])) == len(np.unique(yc_train[validate])) == 2
    fold_rows.append({'fold': fold_number, 'fit_rows': len(fit), 'validation_rows': len(validate),
                      'validation_positive_fraction': yc_train[validate].mean()})
display(pd.DataFrame(fold_rows).round(3))

In [ ]:
cls_pipeline = Pipeline([
    ('variable_bits', VarianceThreshold(threshold=0.0)),
    ('model', LogisticRegression(solver='liblinear', max_iter=1000, random_state=SEED)),
])
cls_search = GridSearchCV(cls_pipeline, {'model__C': [0.1, 1.0]},
                          scoring={'roc_auc': 'roc_auc', 'average_precision': 'average_precision'},
                          refit='roc_auc', cv=cls_folds, n_jobs=1, error_score='raise')
cls_search.fit(Xc_train, yc_train)
cls_cv = pd.DataFrame(cls_search.cv_results_)
display(cls_cv[['param_model__C', 'mean_test_roc_auc', 'std_test_roc_auc',
                'mean_test_average_precision']].round(3))
cls_baseline = DummyClassifier(strategy='prior')
baseline_cls_cv = cross_validate(cls_baseline, Xc_train, yc_train, cv=cls_folds,
                                 scoring={'roc_auc': 'roc_auc', 'average_precision': 'average_precision'},
                                 error_score='raise')
print('Selected C:', cls_search.best_params_['model__C'])
print('Prior baseline validation ROC-AUC:', baseline_cls_cv['test_roc_auc'].mean())
print('Prior baseline validation AP:', baseline_cls_cv['test_average_precision'].mean())

### Which metric answers which question?

With class 1 positive, TP/TN are correct positive/negative classifications; FP/FN are false positives/negatives.

| Metric | Meaning | Important limitation |
|---|---|---|
| Accuracy | $(TP+TN)/n$ | A majority-class rule can look strong when classes are imbalanced |
| Balanced accuracy | Mean of sensitivity and specificity | Gives both classes equal weight; still threshold dependent |
| Precision | $TP/(TP+FP)$ | Depends on class prevalence and threshold |
| Recall (sensitivity) | $TP/(TP+FN)$ | Does not count false positives |
| F1 | $2TP/(2TP+FP+FN)$ | Does not use true negatives; depends on threshold |
| ROC-AUC | Ranking of positive versus negative examples across thresholds | Does not establish probability calibration or performance at a chosen threshold |
| Average precision (AP) | Precision averaged over increments in recall | Depends strongly on positive prevalence; not the trapezoidal PR-curve area |
| Brier score | Mean $(p_i-y_i)^2$, lower is better | Evaluates probability errors, combining calibration and discrimination effects |

For constant prediction scores, ROC-AUC is 0.5 and AP equals the evaluated set's positive fraction (when both classes occur). That can be a surprisingly high AP in BBBP. Here `zero_division=0` reports zero when a threshold metric has a zero denominator; it does not make the absent event scientifically informative. A probability from logistic regression is not automatically calibrated under chemical distribution shift. Sources: [classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics), [AP definition](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html), and [probability calibration](https://scikit-learn.org/stable/modules/calibration.html).

In [ ]:
cls_model = cls_search.best_estimator_
cls_baseline.fit(Xc_train, yc_train)
yc_test = y_cls[cls_test].copy()
assert len(np.unique(yc_test)) == 2, 'ROC-AUC needs both classes in this held-out set.'
probability = cls_model.predict_proba(X_cls[cls_test])[:, 1]
baseline_probability = cls_baseline.predict_proba(X_cls[cls_test])[:, 1]
THRESHOLD = 0.5  # fixed above, not optimized on the test set
prediction = (probability >= THRESHOLD).astype(int)
baseline_prediction = cls_baseline.predict(X_cls[cls_test])

def classification_metrics(y, pred, prob):
    return {'accuracy': accuracy_score(y, pred), 'balanced_accuracy': balanced_accuracy_score(y, pred),
            'precision': precision_score(y, pred, zero_division=0),
            'recall': recall_score(y, pred, zero_division=0), 'F1': f1_score(y, pred, zero_division=0),
            'ROC_AUC': roc_auc_score(y, prob), 'AP': average_precision_score(y, prob),
            'Brier': brier_score_loss(y, prob)}

cls_metrics = pd.DataFrame({
    'selected logistic model': classification_metrics(yc_test, prediction, probability),
    'training prior baseline': classification_metrics(yc_test, baseline_prediction, baseline_probability),
}).T
assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
assert np.isclose(cls_metrics.loc['training prior baseline', 'AP'], yc_test.mean())
cls_results = cls_data.iloc[cls_test][['source_row', 'canonical', 'group', 'y']].copy()
cls_results['probability_class_1'] = probability
cls_results['prediction_at_0.5'] = prediction
cls_results.to_csv(OUT / 'BBBP_test_predictions.csv', index=False)
print('Test class counts [0, 1]:', np.bincount(yc_test, minlength=2))
display(cls_metrics.T.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), layout='constrained')
ConfusionMatrixDisplay.from_predictions(yc_test, prediction, labels=[0, 1], display_labels=['0', '1'],
                                        ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Held-out BBBP labels; threshold 0.5')
precision, recall, _ = precision_recall_curve(yc_test, probability)
axes[1].step(recall, precision, where='post', label=f'Logistic model; AP={cls_metrics.iloc[0]["AP"]:.3f}')
axes[1].axhline(yc_test.mean(), linestyle='--', color='gray', label='Test positive fraction')
axes[1].set(xlabel='Recall for class 1', ylabel='Precision for class 1', xlim=(0, 1), ylim=(0, 1.03),
            title='Precision-recall curve: final diagnostic')
axes[1].legend(fontsize=9, loc='lower left')
fig.savefig(OUT / 'classification_test_diagnostics.png')
plt.show()

### Worked screening interpretation: what does a fixed inspection budget recover?

Keep the selected classifier, its probabilities, and the 0.5 classification results unchanged. Now inspect its **ranking**, using the 20% budget declared before fitting. In this retrospective benchmark exercise, a “positive” means only `p_np = 1`; it is not a successful treatment or a measured outcome from a new screening campaign.

Sort the existing test predictions from highest to lowest class-1 probability, with source row as a deterministic tie-breaker. For the first $k$ records, count their known positives $H(k)$. Then $H(k)/k$ is precision within the inspected group, while $H(k)/N_+$ is the fraction of all test positives recovered. Uniform random selection of $k$ records from this **same finite pool** would have expected count $kN_+/n$.

The curve is a descriptive readout of frozen predictions. We do not search for the “best” budget, revise the probability threshold, exclude difficult molecules, or compare new models. See the [scikit-learn precision/recall definitions](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_curve.html).

In [ ]:
source_order = cls_results['source_row'].to_numpy()
screen_order = np.lexsort((source_order, -probability))
ordered_labels = yc_test[screen_order]
inspected_counts = np.arange(len(yc_test) + 1)
recovered_positives = np.r_[0, np.cumsum(ordered_labels)]
positive_total = int(yc_test.sum())
random_expected = inspected_counts * yc_test.mean()
inspection_count = max(1, int(np.ceil(INSPECTION_FRACTION * len(yc_test))))
positive_at_budget = int(recovered_positives[inspection_count])
negative_at_budget = inspection_count - positive_at_budget
assert recovered_positives[-1] == positive_total
assert positive_at_budget + negative_at_budget == inspection_count
budget_table = pd.DataFrame({
    'group': ['Inspected highest scores', 'Remaining records'],
    'positive_records': [positive_at_budget, positive_total-positive_at_budget],
    'negative_records': [negative_at_budget, len(yc_test)-positive_total-negative_at_budget],
})
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4), layout='constrained')
axes[0].step(inspected_counts, recovered_positives, where='post', label='Frozen classifier ranking')
axes[0].plot(inspected_counts, random_expected, '--', color='0.5', label='Uniform random: expected count')
axes[0].scatter([inspection_count], [positive_at_budget], marker='*', s=130, color='#ba6035',
                label='Predeclared 20% budget', zorder=4)
axes[0].set(xlabel='Number of test records inspected', ylabel='Known positive records recovered',
            title='Retrospective ranking diagnostic', xlim=(0, len(yc_test)), ylim=(0, positive_total+2))
axes[0].legend(fontsize=8)
axes[1].barh(budget_table.group, budget_table.positive_records, color='#28788e', label='Benchmark label 1')
axes[1].barh(budget_table.group, budget_table.negative_records, left=budget_table.positive_records,
              color='#ba6035', label='Benchmark label 0')
axes[1].set(xlabel='Number of records', title='Read the positive base rate')
axes[1].legend(fontsize=8, loc='lower right')
fig.savefig(OUT / 'frozen_screening_budget.png')
plt.show()
display(budget_table)
print(f'Budget: {inspection_count}/{len(yc_test)} records ({inspection_count/len(yc_test):.1%}; rounded up)')
print(f'Precision in inspected group: {positive_at_budget/inspection_count:.3f}')
print(f'Fraction of all test positives recovered: {positive_at_budget/positive_total:.3f}')
print(f'Uniform-random expected positives at this budget: {random_expected[inspection_count]:.2f}')
print(f'Difference from that expectation: {positive_at_budget-random_expected[inspection_count]:+.2f} records; descriptive, not a significance test')
cutoff_score = probability[screen_order[inspection_count-1]]
print(f'Records sharing the last included score exactly: {int(np.sum(probability == cutoff_score))}')
budget_table.to_csv(OUT / 'frozen_screening_budget.csv', index=False)

**Conclusion.** Ranking, a fixed probability threshold, and probability calibration are different questions. A list can contain many positive labels partly because positives are already common in the pool; the random-selection reference makes that base-rate effect visible. Inspect the actual counts and compare them with the reference rather than interpreting a high precision alone as large practical value.

**Next step for a real screening study:** define the outcome, eligible candidate population, experimental budget, costs, and validation plan before making decisions. Evidence on this small curated test set does not establish that the same recovery or class prevalence will hold for future chemical series. The expected random count is a finite-pool reference, not an uncertainty interval.

**Self-check:** could we use this curve to choose a new budget with the most attractive test result, then advertise that point as independently validated? <details><summary>Answer</summary>No. That would adapt a decision to test labels. The revised decision would need selection within appropriate training/validation information and a new independent assessment.</details>

## 6. Interpret the result without overselling it

The selected model and baseline were declared before final evaluation; comparing them now is reporting, not another selection stage. If the baseline is competitive or better, report that outcome. Do not search seeds, exclude poorly predicted compounds, or tune thresholds until the test score improves.

These are single, small scaffold holdouts after explicit exclusions. Their scores have sampling uncertainty and depend on curation, feature settings, label quality, and the split. They do not reproduce published full-dataset benchmark results, establish causation, or measure future clinical performance. More robust research could predefine multiple outer group splits or a genuinely external/time-separated test set, and estimate uncertainty by resampling chemical groups rather than treating all analogs as independent. Such evaluation must include model selection and not just reuse one favorable test set.

The original broad menu of algorithms is useful to recognize, but every algorithm need not be fitted to the same test set:

| Family | Useful intuition | Typical issue to evaluate inside training data |
|---|---|---|
| Linear/ridge/lasso/elastic net | Additive effects in a chosen feature space; different penalties | Scaling, correlated features, nonlinear relationships |
| Nearest neighbors | Transfer labels from nearby training examples | Distance definition, irrelevant dimensions, sparse coverage |
| Trees and random forests | Piecewise rules and averaged rules | Tree complexity, extrapolation, correlated analogs |
| Support vector models | Regularized boundaries/functions, potentially with kernels | Scaling, kernel settings, and training cost |
| Naive Bayes | Combine features under conditional-independence assumptions | Strongly dependent fingerprint bits |
| Gaussian processes | Kernel model with uncertainty conditional on its assumptions | Dense exact fitting scales poorly with sample count; uncertainty need not survive domain shift |

Feature importance or a fitted coefficient is not a causal explanation. Training a larger model is not a substitute for endpoint definition, duplicate control, or appropriate validation. [Chapter 11](Chapter11_Part1.ipynb) introduces neural models; its separately sampled experiments are not a controlled performance comparison with this notebook.

In [ ]:
for name, data in [('logD', reg_data), ('BBBP', cls_data)]:
    data[['source_row', 'smiles', 'canonical', 'y', 'group', 'split']].to_csv(
        OUT / f'{name}_selected_records_and_split.csv', index=False)
reg_metrics.to_csv(OUT / 'regression_test_metrics.csv')
cls_metrics.to_csv(OUT / 'classification_test_metrics.csv')
reg_cv_table.to_csv(OUT / 'regression_validation_metrics.csv', index=False)
manifest = {
    'seed': SEED, 'rdkit': rdBase.rdkitVersion, 'scikit_learn': sklearn.__version__,
    'source_sha256': {str(path): hashlib.sha256(path.read_bytes()).hexdigest()
                      for path in [lipo_path, bbbp_path]},
    'targets': {'Lipophilicity': 'experimental log10 D(octanol/water), pH 7.4',
                'BBBP': 'distributed benchmark p_np label; 1 is positive'},
    'audit': {'Lipophilicity': reg_audit, 'BBBP': cls_audit},
    'descriptor_names': list(descriptor_functions),
    'morgan': {'radius': 2, 'fpSize': 1024, 'includeChirality': True, 'countSimulation': False},
    'grouping': 'RDKit exact Murcko scaffold, no chirality; all acyclic structures one group',
    'outer_split': 'GroupShuffleSplit, test_size=0.25 of groups, fixed seed',
    'model_selection': '3 training-only group folds; MAE for regression, ROC-AUC for classification',
    'selected_regression': reg_cv_table.iloc[reg_search.best_index_]['candidate'],
    'selected_classification_C': float(cls_search.best_params_['model__C']),
    'classification_threshold': THRESHOLD,
    'final_ranking_diagnostic_budget_fraction': INSPECTION_FRACTION,
    'ranking_tie_break': 'original source row; no label-dependent tie-breaking',
}
(OUT / 'evaluation_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved audit trails, source hashes, exact splits, metrics, predictions, and settings to', OUT)

## Exercises and selected answers

1. Why is scaling the full feature table before cross-validation a problem even though `StandardScaler` never sees the labels?
2. Why should a calculated MolLogP descriptor not be named `measured_lipophilicity`? Is including it to predict experimental log D necessarily leakage?
3. A test set contains 90 positives and 10 negatives. Compute accuracy, balanced accuracy, precision, recall, and F1 for predicting class 1 for every record.
4. If a regression model has test $R^2=-0.2$, must its predictions be negatively correlated with the target?
5. Why are conflicting duplicate labels not automatically proof of an experimental mistake? What information would permit a better curation decision?
6. You lower the threshold after inspecting the test confusion matrix. How should you describe the revised score, and what would provide an independent estimate?
7. Does zero scaffold overlap ensure that no highly similar molecules cross the split? Why does keeping all acyclic molecules together create a useful but coarse stress test?

**Selected answers.** (1) The mean and scale encode information from held-out examples; every training fold must learn its own transform. (2) It is a calculated neutral-partition descriptor, while the target is a measured, pH-specific distribution endpoint. A fixed structure-derived descriptor is not leakage by itself. (3) Accuracy 0.90; balanced accuracy 0.50; precision 0.90; recall 1.00; F1 $180/190\approx0.947$. These numbers reveal the limitation of accuracy or F1 alone. (4) No. $R^2$ compares squared errors to a mean predictor and can be negative with positively correlated but biased predictions. (5) Assay conditions, stereochemistry, compound processing, biological system, and measurement noise may differ; seek the original records and endpoint metadata. (6) It is an adapted validation result, not an untouched-test result; choose thresholds inside training/validation data and assess on fresh held-out data. (7) Related chemistry can have different exact scaffolds. The acyclic group prevents within-group transfer but lumps chemically diverse molecules together, so its location can strongly affect results.